# CKD Prediction — Preprocessing & Model Experiments
### Pipeline: KNNImputer + Ensemble Classifiers + GridSearchCV

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.impute import KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split, GridSearchCV, StratifiedKFold
)
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, VotingClassifier, StackingClassifier
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score,
    confusion_matrix, classification_report
)

os.makedirs('../report', exist_ok=True)
RANDOM_STATE = 42
print("Libraries loaded ✓")

Libraries loaded ✓


In [2]:
df = pd.read_csv("data/kidney_disease.csv")
print("Shape:", df.shape)
df.head(3)

Shape: (399, 25)


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,class
0,48.0,80.0,1.02,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,...,44.0,7800.0,5.2,yes,yes,no,good,no,no,ckd
1,7.0,50.0,1.02,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,...,38.0,6000.0,NaN,no,no,no,good,no,no,ckd
2,62.0,80.0,1.01,2.0,3.0,normal,normal,notpresent,notpresent,423.0,...,31.0,7500.0,NaN,no,yes,no,poor,no,yes,ckd


In [3]:
# Drop rows where target is missing
df.dropna(subset=['class'], inplace=True)
print("After dropping missing target:", df.shape)

# Encode categorical features
cat_cols = ['rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane']
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = df[col].fillna('missing')
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Encode target
le_target = LabelEncoder()
y = le_target.fit_transform(df['class'])
print("\nTarget encoding:", dict(zip(le_target.classes_, 
    le_target.transform(le_target.classes_))))

X = df.drop(columns=['class'])
print("\nFeatures shape:", X.shape)
print("Target shape:", y.shape)

After dropping missing target: (399, 25)
rbc: {'abnormal': np.int64(0), 'missing': np.int64(1), 'normal': np.int64(2)}
pc: {'abnormal': np.int64(0), 'missing': np.int64(1), 'normal': np.int64(2)}
pcc: {'missing': np.int64(0), 'notpresent': np.int64(1), 'present': np.int64(2)}
ba: {'missing': np.int64(0), 'notpresent': np.int64(1), 'present': np.int64(2)}
htn: {'missing': np.int64(0), 'no': np.int64(1), 'yes': np.int64(2)}
dm: {'missing': np.int64(0), 'no': np.int64(1), 'yes': np.int64(2)}
cad: {'missing': np.int64(0), 'no': np.int64(1), 'yes': np.int64(2)}
appet: {'good': np.int64(0), 'missing': np.int64(1), 'poor': np.int64(2)}
pe: {'missing': np.int64(0), 'no': np.int64(1), 'yes': np.int64(2)}
ane: {'missing': np.int64(0), 'no': np.int64(1), 'yes': np.int64(2)}

Target encoding: {'ckd': np.int64(0), 'notckd': np.int64(1)}

Features shape: (399, 24)
Target shape: (399,)


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)
print("\nTrain class distribution:", 
    dict(zip(*np.unique(y_train, return_counts=True))))
print("Test class distribution:", 
    dict(zip(*np.unique(y_test, return_counts=True))))

Train size: (319, 24)
Test size: (80, 24)

Train class distribution: {np.int64(0): np.int64(200), np.int64(1): np.int64(119)}
Test class distribution: {np.int64(0): np.int64(50), np.int64(1): np.int64(30)}


In [5]:
# Pipeline ensures KNNImputer fits ONLY on training data
# No data leakage possible

pipe = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5)),
    ("clf", RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

print("Pipeline created ✓")
print(pipe)

Pipeline created ✓
Pipeline(steps=[('imputer', KNNImputer()),
                ('clf', RandomForestClassifier(n_jobs=-1, random_state=42))])


In [6]:
param_grid = {
    "imputer__n_neighbors": [3, 5, 7],
    "clf__n_estimators":    [100, 200, 300],
    "clf__max_depth":       [5, 10, 20, None],
    "clf__min_samples_split": [2, 5, 10],
    "clf__min_samples_leaf":  [1, 2, 4],
    "clf__max_features":    ["sqrt", "log2"],
}

print("Total combinations:", 
    3*3*4*3*3*2, "parameter sets")

Total combinations: 648 parameter sets


In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)
print("\nBest params:", grid.best_params_)
print("Best CV f1_macro:", round(grid.best_score_, 4))

Fitting 5 folds for each of 648 candidates, totalling 3240 fits

Best params: {'clf__max_depth': 5, 'clf__max_features': 'sqrt', 'clf__min_samples_leaf': 1, 'clf__min_samples_split': 2, 'clf__n_estimators': 100, 'imputer__n_neighbors': 3}
Best CV f1_macro: 0.9864


In [8]:
y_pred = grid.predict(X_test)

print("=== Test Set Results (Random Forest) ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, average='macro'):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred, average='macro'):.4f}")
print(f"F1 Macro:  {f1_score(y_test, y_pred, average='macro'):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, 
    target_names=le_target.classes_))

=== Test Set Results (Random Forest) ===
Accuracy:  0.9875
Precision: 0.9902
Recall:    0.9833
F1 Macro:  0.9866

Classification Report:
              precision    recall  f1-score   support

         ckd       0.98      1.00      0.99        50
      notckd       1.00      0.97      0.98        30

    accuracy                           0.99        80
   macro avg       0.99      0.98      0.99        80
weighted avg       0.99      0.99      0.99        80



## Best Hyperparameters Found

Copy these into ml_v2.ipynb for final model training.

GridSearchCV used:
- 5-fold Stratified Cross Validation
- f1_macro scoring (handles class imbalance)
- KNNImputer inside Pipeline (no leakage)